# 05.1 — Model Analytics & Deep Diagnostics (Bias-Free)

Bu notebook, modelin (son fold) derinlemesine istatistiksel MR'ını çeker.

**Önceki versiyondan farklar (look-ahead bias düzeltmeleri):**
- Rank IC artık `tb_return` yerine **realized forward return** kullanır (bias-free)
- Latent space ve SHAP analizi sadece **OOS test verisi** üzerinde yapılır
- Feature isimleri veri sırasıyla `[latent, handcrafted, regime]` olarak eşleştirilir
- Tüm fold'lar üzerinde **per-fold IC cross-validation** eklendi

In [ ]:
# Gerekli kütüphaneleri kuralım
!pip install -q xgboost shap scikit-learn matplotlib seaborn pandas numpy
!pip install -q pyarrow fastparquet hmmlearn ccxt PyWavelets numba

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys, json, pickle
REPO_DIR = '/content/scalp2_repo'
if os.path.exists(os.path.join(REPO_DIR, '.git')):
    !git -C {REPO_DIR} pull --ff-only
else:
    !git clone https://github.com/umutergul74/Scalp2.git {REPO_DIR}

sys.path.insert(0, REPO_DIR)

import numpy as np
import pandas as pd
import torch

from scalp2.config import load_config
config = load_config(f'{REPO_DIR}/config.yaml')

DATA_DIR = '/content/drive/MyDrive/scalp2/data/processed'
CHECKPOINT_DIR = '/content/drive/MyDrive/scalp2/checkpoints'

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import xgboost as xgb
from scipy.stats import spearmanr
from sklearn.calibration import calibration_curve
from sklearn.metrics import brier_score_loss, accuracy_score, f1_score, precision_score, recall_score
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
from scalp2.utils.serialization import load_fold_artifacts
from scalp2.models.hybrid import HybridEncoder
from scalp2.data.dataset import ScalpDataset
from torch.utils.data import DataLoader
from torch.amp import autocast

sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

## 1. Veri & Artifacts Yükleme

In [ ]:
# Walk-forward predictions yükle (tüm fold'lar)
with open(f'{DATA_DIR}/wf_predictions.pkl', 'rb') as f:
    wf_predictions = pickle.load(f)

TARGET_FOLD = wf_predictions[-1]['fold_idx']  # Son fold
print(f'Toplam fold sayısı: {len(wf_predictions)}')
print(f'Analiz Edilen Model: Fold {TARGET_FOLD} (son fold)')

# Labeled dataset yükle
df = pd.read_parquet(f'{DATA_DIR}/BTC_USDT_labeled.parquet')
with open(f'{DATA_DIR}/feature_columns.json', 'r') as f:
    feature_cols = json.load(f)

features_array = df[feature_cols].values
labels_array = df['tb_label_cls'].values
close_prices = df['close'].values.astype(np.float64)

# Son fold'un artifacts'larını yükle
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
artifacts = load_fold_artifacts(CHECKPOINT_DIR, TARGET_FOLD, device=device)

n_features = len(feature_cols)
model_tcn = HybridEncoder(n_features, config.model)
model_tcn.load_state_dict(artifacts['model_state'])
model_tcn.eval()
model_tcn.to(device)

scaler = artifacts['scaler']
regime_detector = artifacts.get('regime_detector')
top_indices = artifacts.get('top_feature_indices')
seq_len = config.model.seq_len

print(f'Dataset: {len(df)} bars, {len(feature_cols)} features')
print(f'Seq len: {seq_len}, Top indices: {len(top_indices) if top_indices is not None else "N/A"}')
print('Artifacts yüklendi. Model hazır.')

## 2. Kalibrasyon, IC & Sınıflandırma Metrikleri (Bias-Free)

**KRİTİK DÜZELTME:** Eski versiyon `tb_return` kullanıyordu — bu değer geleceğin fiyat hareketinden türetildiği için yapay yüksek IC üretiyordu.

Artık **realized 1-bar forward return** (`close[t+1]/close[t] - 1`) kullanılıyor. Bu, modelin sinyal ürettiği anda bilinmesi imkansız olan gelecek getirisini temsil eder.

In [ ]:
fold_data = next((f for f in wf_predictions if f['fold_idx'] == TARGET_FOLD), None)

if fold_data is None:
    raise ValueError(f'Fold {TARGET_FOLD} bulunamadı!')

probs = fold_data['test_probabilities']
labels = fold_data['test_labels']

# ============================================================
# BIAS-FREE Realized Return Hesabı
# ============================================================
# Test periyodundaki barların gerçek 1-bar-forward return'ü
# Bu değer sinyal anında bilinmez → look-ahead bias yok
test_bar_start = fold_data['test_start'] + seq_len
test_bar_end = test_bar_start + len(labels)

# 1-bar forward: close[t+1] / close[t] - 1
test_close = close_prices[test_bar_start:test_bar_end + 1]  # +1 for shift
realized_returns_1bar = (test_close[1:] / test_close[:-1]) - 1.0
realized_returns_1bar = realized_returns_1bar[:len(labels)]  # truncate to match

# 10-bar forward (horizon-aligned): close[t+10] / close[t] - 1
max_hold = config.labeling.max_holding_bars
test_close_extended = close_prices[test_bar_start:test_bar_end + max_hold]
if len(test_close_extended) >= len(labels) + max_hold:
    realized_returns_10bar = (
        test_close_extended[max_hold:max_hold + len(labels)] /
        test_close_extended[:len(labels)]
    ) - 1.0
else:
    # Truncate if not enough bars
    n_valid = len(test_close_extended) - max_hold
    realized_returns_10bar = (
        test_close_extended[max_hold:max_hold + n_valid] /
        test_close_extended[:n_valid]
    ) - 1.0

print(f'Test periyodu: bar {test_bar_start} → {test_bar_end}')
print(f'Test tarihleri: {df.index[test_bar_start]} → {df.index[min(test_bar_end, len(df)-1)]}')
print(f'Test sample sayısı: {len(labels)}')
print(f'Realized returns (1-bar): {len(realized_returns_1bar)} samples')
print(f'Realized returns ({max_hold}-bar): {len(realized_returns_10bar)} samples')

# ============================================================
# Kalibrasyon (Brier Score & Reliability Diagram)
# ============================================================
is_long = (labels == 2).astype(int)
b_score = brier_score_loss(is_long, probs[:, 2])
print(f'\n▶ Brier Score (Kalibrasyon Hatası - Long Yönü): {b_score:.4f}')

prob_true, prob_pred = calibration_curve(is_long, probs[:, 2], n_bins=10)

plt.figure(figsize=(8, 6))
plt.plot(prob_pred, prob_true, marker='o', linewidth=2, label=f'XGBoost Fold {TARGET_FOLD:03d} (Long)')
plt.plot([0, 1], [0, 1], linestyle='--', color='gray', label='Mükemmel Olasılık (1:1)')
plt.title('Reliability Diagram (Kalibrasyon Eğrisi) - OOS Test', fontsize=14)
plt.xlabel('Tahmin Edilen Long Olasılığı', fontsize=12)
plt.ylabel('Gerçekleşen Başarı (Long Hedefe Ulaşma)', fontsize=12)
plt.legend()
plt.savefig('diag_calibration.png', bbox_inches='tight')
plt.show()

# ============================================================
# Rank IC — BIAS-FREE (realized return kullanarak)
# ============================================================
net_prob = probs[:, 2] - probs[:, 0]  # Long olasılığı - Short olasılığı

# 1-bar IC
ic_1bar, p_1bar = spearmanr(net_prob[:len(realized_returns_1bar)], realized_returns_1bar)
print(f'\n▶ Rank IC (1-bar forward, bias-free): {ic_1bar:.4f} (p={p_1bar:.4f})')

# 10-bar IC (horizon-aligned)
ic_10bar, p_10bar = spearmanr(net_prob[:len(realized_returns_10bar)], realized_returns_10bar)
print(f'▶ Rank IC ({max_hold}-bar forward, bias-free): {ic_10bar:.4f} (p={p_10bar:.4f})')

if ic_1bar > 0.03:
    print('✅ Pozitif IC — model gerçek fiyat yönünü tahmin edebiliyor.')
elif ic_1bar > 0.0:
    print('⚠️ Zayıf pozitif IC — marjinal edge olabilir.')
else:
    print('❌ Negatif/sıfır IC — model fiyat yönünü tahmin edemiyor.')

# ============================================================
# Classification Metrics
# ============================================================
y_pred = np.argmax(probs, axis=1)
cls_names = ['Short (0)', 'Hold (1)', 'Long (2)']
p_per = precision_score(labels, y_pred, labels=[0,1,2], average=None, zero_division=0)
r_per = recall_score(labels, y_pred, labels=[0,1,2], average=None, zero_division=0)
f_per = f1_score(labels, y_pred, labels=[0,1,2], average=None, zero_division=0)

acc = accuracy_score(labels, y_pred)
macro_f1 = f1_score(labels, y_pred, average='macro', zero_division=0)
weighted_f1 = f1_score(labels, y_pred, average='weighted', zero_division=0)

print(f'\n▶ CLASSIFICATION METRICS (OOS Test - Fold {TARGET_FOLD}):')
print(f'  Overall Accuracy: {acc:.4f}')
print(f'  ---')
for i, name in enumerate(cls_names):
    print(f'  {name}: Prec={p_per[i]:.4f}  Rec={r_per[i]:.4f}  F1={f_per[i]:.4f}')
print(f'  ---')
print(f'  Macro Avg F1: {macro_f1:.4f}')
print(f'  Weighted Avg F1: {weighted_f1:.4f}')

# Metrikleri dosyaya yaz
with open('diag_metrics.txt', 'w') as mf:
    mf.write(f'=== BIAS-FREE Model Diagnostics (Fold {TARGET_FOLD}) ===\n')
    mf.write(f'Brier Score: {b_score:.4f}\n')
    mf.write(f'Rank IC (1-bar, bias-free): {ic_1bar:.4f} (p={p_1bar:.4f})\n')
    mf.write(f'Rank IC ({max_hold}-bar, bias-free): {ic_10bar:.4f} (p={p_10bar:.4f})\n')
    mf.write(f'Accuracy: {acc:.4f}\n')
    mf.write(f'--- Per-Class Metrics ---\n')
    for i, name in enumerate(cls_names):
        mf.write(f'{name}: Prec={p_per[i]:.4f}  Rec={r_per[i]:.4f}  F1={f_per[i]:.4f}\n')
    mf.write(f'Macro Avg F1: {macro_f1:.4f}\n')
    mf.write(f'Weighted Avg F1: {weighted_f1:.4f}\n')

## 3. Per-Fold IC Cross-Validation (Tüm Fold'lar)

Tek bir fold'a bakmak yanıltıcı olabilir. Modelin gerçek gücünü anlamak için **tüm fold'ların test setleri** üzerinde ayrı ayrı IC hesaplıyoruz.

In [ ]:
fold_ics_1bar = []
fold_ics_10bar = []
fold_accs = []
fold_f1s = []

for fold_d in wf_predictions:
    fidx = fold_d['fold_idx']
    f_probs = fold_d['test_probabilities']
    f_labels = fold_d['test_labels']
    n_test = len(f_labels)

    f_test_start = fold_d['test_start'] + seq_len
    f_test_end = f_test_start + n_test

    # 1-bar forward return
    if f_test_end + 1 <= len(close_prices):
        f_close = close_prices[f_test_start:f_test_end + 1]
        f_ret_1 = (f_close[1:] / f_close[:-1]) - 1.0
        f_ret_1 = f_ret_1[:n_test]
        f_net = f_probs[:len(f_ret_1), 2] - f_probs[:len(f_ret_1), 0]
        if len(f_net) > 10:
            ic, _ = spearmanr(f_net, f_ret_1)
            fold_ics_1bar.append((fidx, ic))

    # 10-bar forward return
    if f_test_end + max_hold <= len(close_prices):
        f_close_ext = close_prices[f_test_start:f_test_end + max_hold]
        f_ret_10 = (f_close_ext[max_hold:max_hold + n_test] / f_close_ext[:n_test]) - 1.0
        f_net10 = f_probs[:len(f_ret_10), 2] - f_probs[:len(f_ret_10), 0]
        if len(f_net10) > 10:
            ic10, _ = spearmanr(f_net10, f_ret_10)
            fold_ics_10bar.append((fidx, ic10))

    # Accuracy & F1
    f_pred = np.argmax(f_probs, axis=1)
    fold_accs.append((fidx, accuracy_score(f_labels, f_pred)))
    fold_f1s.append((fidx, f1_score(f_labels, f_pred, average='macro', zero_division=0)))

# Sonuçları raporla
ics_1 = [x[1] for x in fold_ics_1bar]
ics_10 = [x[1] for x in fold_ics_10bar]
accs = [x[1] for x in fold_accs]
f1s = [x[1] for x in fold_f1s]

print('=' * 60)
print('  PER-FOLD CROSS-VALIDATION SONUÇLARI (BIAS-FREE)')
print('=' * 60)
print(f'  Toplam fold: {len(wf_predictions)}')
print(f'  ---')
print(f'  Rank IC (1-bar):  Medyan={np.median(ics_1):.4f}, Ort={np.mean(ics_1):.4f}, Std={np.std(ics_1):.4f}')
print(f'                    Min={np.min(ics_1):.4f}, Max={np.max(ics_1):.4f}')
print(f'  Rank IC ({max_hold}-bar): Medyan={np.median(ics_10):.4f}, Ort={np.mean(ics_10):.4f}, Std={np.std(ics_10):.4f}')
print(f'                    Min={np.min(ics_10):.4f}, Max={np.max(ics_10):.4f}')
print(f'  ---')
print(f'  Accuracy:  Medyan={np.median(accs):.4f}, Ort={np.mean(accs):.4f}, Std={np.std(accs):.4f}')
print(f'  Macro F1:  Medyan={np.median(f1s):.4f}, Ort={np.mean(f1s):.4f}, Std={np.std(f1s):.4f}')
print('=' * 60)

# IC pozitif olan fold yüzdesi
pct_pos_1 = 100 * np.mean(np.array(ics_1) > 0)
pct_pos_10 = 100 * np.mean(np.array(ics_10) > 0)
print(f'\n  IC > 0 olan fold oranı (1-bar): {pct_pos_1:.0f}%')
print(f'  IC > 0 olan fold oranı ({max_hold}-bar): {pct_pos_10:.0f}%')

if np.median(ics_1) > 0.02:
    print('\n  ✅ Model tutarlı pozitif IC üretiyor — gerçek prediktif edge var.')
elif np.median(ics_1) > 0.0:
    print('\n  ⚠️ Marjinal IC — edge çok zayıf veya belirli dönemlere özel.')
else:
    print('\n  ❌ Medyan IC ≤ 0 — model fiyat yönünü tutarlı tahmin edemiyor.')

# Görselleştir
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].bar([x[0] for x in fold_ics_1bar], ics_1,
            color=['#4CAF50' if v > 0 else '#F44336' for v in ics_1], alpha=0.7)
axes[0].axhline(0, color='black', linewidth=0.5)
axes[0].axhline(np.median(ics_1), color='blue', linestyle='--', alpha=0.5, label=f'Medyan: {np.median(ics_1):.4f}')
axes[0].set_title('Per-Fold Rank IC (1-bar forward return)', fontsize=12)
axes[0].set_xlabel('Fold Index')
axes[0].set_ylabel('Spearman IC')
axes[0].legend()

axes[1].bar([x[0] for x in fold_ics_10bar], ics_10,
            color=['#4CAF50' if v > 0 else '#F44336' for v in ics_10], alpha=0.7)
axes[1].axhline(0, color='black', linewidth=0.5)
axes[1].axhline(np.median(ics_10), color='blue', linestyle='--', alpha=0.5, label=f'Medyan: {np.median(ics_10):.4f}')
axes[1].set_title(f'Per-Fold Rank IC ({max_hold}-bar forward return)', fontsize=12)
axes[1].set_xlabel('Fold Index')
axes[1].set_ylabel('Spearman IC')
axes[1].legend()

plt.suptitle('Fold Bazlı IC Dağılımı (Bias-Free)', fontsize=14)
plt.tight_layout()
plt.savefig('diag_fold_ic.png', bbox_inches='tight')
plt.show()

# Metriklere ekle
with open('diag_metrics.txt', 'a') as mf:
    mf.write(f'\n--- Per-Fold Cross-Validation ---\n')
    mf.write(f'Median IC (1-bar): {np.median(ics_1):.4f}\n')
    mf.write(f'Median IC ({max_hold}-bar): {np.median(ics_10):.4f}\n')
    mf.write(f'IC > 0 folds (1-bar): {pct_pos_1:.0f}%\n')
    mf.write(f'Median Accuracy: {np.median(accs):.4f}\n')
    mf.write(f'Median Macro F1: {np.median(f1s):.4f}\n')

## 4. Deep Learning Latent Space Analizi (OOS Test Verisi)

**DÜZELTME:** Eski versiyon `df.iloc[-5000:-1]` kullanıyordu — bu train verisini içeriyordu.

Artık sadece Fold'un **OOS test periyodu** üzerinde latent vektörler çıkarılıyor.

In [ ]:
# OOS test verisinin indekslerini al
oos_start = fold_data['test_start']
oos_end = fold_data['test_start'] + len(fold_data['test_labels']) + seq_len
oos_end = min(oos_end, len(df))  # Sınır kontrolü

print(f'OOS Test periyodu: index [{oos_start}:{oos_end}]')
print(f'OOS Tarih aralığı: {df.index[oos_start]} → {df.index[oos_end - 1]}')

# OOS verisini hazırla
oos_feat = features_array[oos_start:oos_end]
oos_labels = labels_array[oos_start:oos_end]

# Fold'un scaler'ı ile scale et (train'de fit edilmiş)
oos_scaled = scaler.transform(oos_feat).astype(np.float32)
oos_scaled = np.nan_to_num(oos_scaled, nan=0.0, posinf=0.0, neginf=0.0)

# Latent vektörleri çıkar
dummy_returns = np.zeros(len(oos_labels), dtype=np.float32)
dataset = ScalpDataset(oos_scaled, oos_labels, dummy_returns, seq_len)
loader = DataLoader(dataset, batch_size=512, shuffle=False)

latents = []
amp_device = 'cuda' if device.type == 'cuda' else 'cpu'
amp_enabled = device.type == 'cuda'
with torch.no_grad():
    for x, _, _ in loader:
        x = x.to(device)
        with autocast(amp_device, enabled=amp_enabled):
            _, latent = model_tcn(x)
        latents.append(latent.cpu().numpy())

X_latent = np.vstack(latents)
y_latent_labels = oos_labels[seq_len:]  # Align with latent output

# Regime probs (forward-only = no look-ahead)
oos_df = df.iloc[oos_start + seq_len:oos_end]
if regime_detector is not None:
    regime_probs = regime_detector.predict_proba_online(oos_df)
else:
    regime_probs = np.full((len(oos_df), 3), 1/3, dtype=np.float32)

# Handcrafted features (top-K selected)
X_base = oos_scaled[seq_len:]
if top_indices is not None and len(top_indices) > 0:
    X_base_selected = X_base[:, top_indices]
    hc_names = [feature_cols[i] for i in top_indices]
else:
    X_base_selected = X_base
    hc_names = feature_cols

# Truncate to shortest
min_len = min(len(X_latent), len(X_base_selected), len(regime_probs))
X_latent = X_latent[:min_len]
X_base_selected = X_base_selected[:min_len]
regime_probs = regime_probs[:min_len]
y_latent_labels = y_latent_labels[:min_len]

# DOĞRU SIRALAMA: [latent | handcrafted | regime] — meta_learner.py ile aynı
X_xgb = np.hstack([X_latent, X_base_selected, regime_probs])

# Feature isimleri DOĞRU SIRALAMA ile
feature_names_xgb = (
    [f'latent_{i}' for i in range(X_latent.shape[1])] +
    hc_names +
    ['regime_bull', 'regime_bear', 'regime_choppy']
)

print(f'OOS Feature matrisi: {X_xgb.shape}')
print(f'Feature name sırası: [{X_latent.shape[1]} latent] + [{len(hc_names)} handcrafted] + [3 regime]')
print(f'  = {len(feature_names_xgb)} toplam (veri ile eşleşiyor: {X_xgb.shape[1] == len(feature_names_xgb)})')

In [ ]:
# PCA & t-SNE (OOS verisi üzerinde)
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_latent)
pca_var = np.sum(pca.explained_variance_ratio_)
print(f'▶ PCA İzah Edilen Varyans: {pca_var:.4f}')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
scatter1 = ax1.scatter(X_pca[:, 0], X_pca[:, 1], c=y_latent_labels, cmap='viridis', alpha=0.5, s=15)
ax1.set_title('PCA İzdüşümü (Doğrusal Latent Ayrımlar)')
ax1.set_xlabel('PCA 1')
ax1.set_ylabel('PCA 2')

print('t-SNE işlemi yürütülüyor (Yaklaşık 10-20 sn)...')
tsne = TSNE(n_components=2, perplexity=30, n_iter=1000, random_state=42)
X_tsne = tsne.fit_transform(X_latent)

scatter2 = ax2.scatter(X_tsne[:, 0], X_tsne[:, 1], c=y_latent_labels, cmap='viridis', alpha=0.5, s=15)
ax2.set_title('t-SNE İzdüşümü (Non-Linear Manifold)')
ax2.set_xlabel('t-SNE 1')
ax2.set_ylabel('t-SNE 2')

plt.suptitle(f'Deep Learning (TCN+GRU) Öznitelik Ayrıştırma — OOS Test (Fold {TARGET_FOLD})', fontsize=15)
plt.savefig('diag_latent.png', bbox_inches='tight')
plt.show()

with open('diag_metrics.txt', 'a') as mf:
    mf.write(f'PCA Explained Var: {pca_var:.4f}\n')

print('\n✨ YORUM: Noktalar renklerine göre adalara bölünüyorsa model ayrıştırma işini çözmüştür.')
print('   Tam bir gürültü eğrisi (yuvarlak duman) ise DL kısmı zayıf kalmıştır.')

## 5. XGBoost Karar Faktörleri (SHAP — OOS Verisi)

**DÜZELTME:** SHAP artık sadece OOS test verisi üzerinde hesaplanıyor ve feature isimleri doğru sırayla eşleştiriliyor.

In [ ]:
# XGBoost modelini yükle
xgb_path = f'{CHECKPOINT_DIR}/xgb_fold_{TARGET_FOLD:03d}.json'
if not os.path.exists(xgb_path):
    # Alternatif konum
    xgb_path = f'{CHECKPOINT_DIR}/fold_{TARGET_FOLD:03d}/xgb_fold_{TARGET_FOLD:03d}.json'

booster = xgb.Booster()
booster.load_model(xgb_path)
print(f'XGBoost model yüklendi: {xgb_path}')

# SHAP — OOS verisi üzerinde (max 2000 sample)
n_shap = min(2000, len(X_xgb))
X_shap = X_xgb[:n_shap]

print(f'SHAP Değerleri Hesaplanıyor ({n_shap} OOS sample)...')
explainer = shap.TreeExplainer(booster)
shap_values = explainer.shap_values(X_shap)

if isinstance(shap_values, list):
    shap_long = shap_values[2]  # Long sınıfı
elif len(np.array(shap_values).shape) == 3:
    shap_long = shap_values[:, :, 2]
else:
    shap_long = shap_values

plt.figure(figsize=(10, 8))
plt.title(f'SHAP Summary (Long Yönü) - Fold {TARGET_FOLD} (OOS)', fontsize=16)
shap.summary_plot(shap_long, X_shap, feature_names=feature_names_xgb, show=False)
plt.tight_layout()
plt.savefig('diag_shap.png', bbox_inches='tight')
plt.show()

# Top 3 SHAP features
top3_idx = np.abs(shap_long).mean(0).argsort()[::-1][:3]
top3_names = [feature_names_xgb[i] for i in top3_idx]
print(f'\nSHAP Top 3 (Long): {", ".join(top3_names)}')

with open('diag_metrics.txt', 'a') as mf:
    mf.write(f'SHAP Top 3 (Long): {", ".join(top3_names)}\n')

## 6. Çıktıları Paketle

In [ ]:
import shutil

print('Çıktılar paketleniyor...')
os.makedirs('diag_export', exist_ok=True)
files_to_copy = ['diag_metrics.txt', 'diag_calibration.png', 'diag_latent.png', 'diag_shap.png', 'diag_fold_ic.png']

for f in files_to_copy:
    if os.path.exists(f):
        shutil.copy(f, f'diag_export/{f}')
        print(f'  {f} eklendi.')

shutil.make_archive('diagnostic_results', 'zip', 'diag_export')
print('\n✅ BİTTİ! Colab sol menüsünden diagnostic_results.zip dosyasını indirebilirsiniz.')

# Son metrikleri göster
print('\n' + '=' * 60)
print(open('diag_metrics.txt').read())
print('=' * 60)